In [1]:
# %load_ext autoreload
# %autoreload 2
import pygsti as pg
import numpy as np
import scipy.linalg as la

from pygsti.baseobjs import qubitgraph as _qgraph
from pygsti.protocols import StandardGSTDesign
from pygsti.protocols.xfgst_edesign import make_xfgst_design
from pygsti.modelpacks import smq1Q_XYI
from pygsti.processors import QubitProcessorSpec
from pygsti.baseobjs import QubitSpace

from pygsti.data import simulate_data

In [2]:

def make_nq_spam(num_qubits):
    from pygsti.modelmembers.states import ComposedState, ComputationalBasisState
    from pygsti.modelmembers.povms import ComposedPOVM
    from pygsti.modelmembers.operations import LindbladErrorgen, ExpErrorgenOp
    from pygsti.baseobjs.errorgenbasis import CompleteElementaryErrorgenBasis
    state_space = QubitSpace(num_qubits)
    max_weights = {'H':1, 'S':1, 'C':1, 'A':1}
    egbn_H_only = CompleteElementaryErrorgenBasis('PP', state_space, ('H',), max_weights)

    rho_errgen_rates = {ell: 0.0 for ell in egbn_H_only.labels}
    rho_lindblad = LindbladErrorgen.from_elementary_errorgens(rho_errgen_rates, parameterization='H', state_space=state_space, evotype='densitymx')
    rho_errorgen = ExpErrorgenOp(rho_lindblad)
    rho_ideal    = ComputationalBasisState([0]*num_qubits)
    rho          = ComposedState(rho_ideal, rho_errorgen)

    M_errgen_rates = {ell: 0.0 for ell in egbn_H_only.labels}
    M_lindblad = LindbladErrorgen.from_elementary_errorgens(M_errgen_rates, parameterization='H', state_space=state_space, evotype='densitymx')
    M_errorgen = ExpErrorgenOp(M_lindblad)
    M = ComposedPOVM(M_errorgen)

    return rho, M


def make_nq_XYIECR_target_model(num_qubits):
    from pygsti.models import modelconstruction as pgmc
    from pygsti.baseobjs.errorgenbasis import CompleteElementaryErrorgenBasis
    ps_geometry = _qgraph.QubitGraph.common_graph(
        num_qubits, geometry='line',
        directed=True, all_directions=True,
        qubit_labels=tuple(range(num_qubits))
    )
    u_ecr = 1/np.sqrt(2)*np.array([[0,0,1,1j],[0,0,1j,1],[1,-1j,0,0],[-1j,1,0,0]])
    gatenames = ['Gxpi2', 'Gypi2', 'Gi', 'Gii',  'Gecr']
    ps = QubitProcessorSpec(
        num_qubits=num_qubits,
        gate_names=gatenames,
        nonstd_gate_unitaries={'Gecr': u_ecr, 'Gii': np.eye(4)},
        geometry=ps_geometry
    )
    gateerrs = dict()
    egb1 = CompleteElementaryErrorgenBasis('PP', QubitSpace(1), ('H','S'), default_label_type='local')
    for gn in gatenames[:-1]:
        gateerrs[gn] = {ell: 0 for ell in egb1.labels}
    egb2 = CompleteElementaryErrorgenBasis('PP', QubitSpace(2), ('H','S'), default_label_type='local')
    gateerrs['Gecr'] = {ell: 0 for ell in egb2.labels}
    gateerrs['Gii'] = gateerrs['Gecr']

    tmn = pgmc.create_crosstalk_free_model(ps, lindblad_error_coeffs=gateerrs)

    rho, M = make_nq_spam(num_qubits)
    tmn.prep_blks['layers']['rho0'] = rho
    tmn.povm_blks['layers']['Mdefault'] = M
    tmn._rebuild_paramvec()
    
    return tmn


def make_2q_XYIECR_cptplnd():
    from pygsti.models import modelconstruction as pgmc
    u_ecr = 1/np.sqrt(2)*np.array([[0,0,1,1j],[0,0,1j,1],[1,-1j,0,0],[-1j,1,0,0]])
    gatenames = ['Gxpi2', 'Gypi2', 'Gii',  'Gecr']
    ps = QubitProcessorSpec(
        num_qubits=2,
        gate_names=gatenames,
        nonstd_gate_unitaries={'Gecr': u_ecr, 'Gii': np.eye(4)},
        availability={
            'Gxpi2': [(0,), (1,)],
            'Gypi2': [(0,), (1,)],
            'Gecr' : [(0, 1), (1, 0)],
            'Gii'  : [(0, 1)]
        }
    )
    tm2 = pgmc.create_explicit_model(ps)
    tm2.convert_members_inplace('CPTPLND')
    return tm2


def make_2q_edesign(target_model2Q, mml):
    from pygsti.algorithms import fiducialselection as fidsel
    from pygsti.algorithms import germselection as germsel
    prep_fiducials2Q, meas_fiducials2Q = fidsel.find_fiducials(target_model2Q, algorithm='greedy', candidate_fid_counts=4)
    germs2Q = germsel.find_germs(target_model2Q, randomize=False, algorithm='greedy', assume_real=True, float_type=np.double,
                                candidate_germ_counts={5: 'all upto'}, mode='compactEVD', verbosity=2)
    temp = [c for c in germs2Q]
    germs2Q = []
    for c in temp:
        new_c = c.copy(editable=True)
        new_c.line_labels = (0,1)
        new_c.done_editing()
        germs2Q.append(new_c)
    Ls = [2**k for k in range(round(np.log2(mml)+1))]
    from pygsti.algorithms.fiducialpairreduction import find_sufficient_fiducial_pairs_per_germ
    fidpairs = find_sufficient_fiducial_pairs_per_germ(target_model2Q, prep_fiducials2Q, meas_fiducials2Q, germs2Q)
    edesign2Q = StandardGSTDesign(target_model2Q, prep_fiducials2Q, meas_fiducials2Q, germs2Q, Ls, fiducial_pairs=fidpairs)
    return edesign2Q


In [3]:
maxmaxlen = 8
nq = 3
oneq_gstdesign = smq1Q_XYI.create_gst_experiment_design(maxmaxlen)

In [4]:
twoq_target = make_2q_XYIECR_cptplnd()

/Users/rjmurr/Documents/pg-xfgst/repo/pygsti/modelmembers/povms/__init__.py:622: FutureWarning: `rcond` parameter will change to the default of machine precision times ``max(M, N)`` where M and N are the input matrix dimensions.
To use the future default and silence this warning we advise to pass `rcond=None`, to keep using the old, explicitly pass `rcond=-1`.
  errgen_vec = _np.linalg.lstsq(phys_directions, soln.x)[0]


In [5]:
twoq_target._print_gpindices(0)

PRINTING MODEL GPINDICES!!!
>>> rho0 [ComposedState]: 240 params, indices=slice(0, 240, None), 2 sub-members
>>> Mdefault [ComposedPOVM]: 240 params, indices=slice(240, 480, None), 2 sub-members
>>> Gxpi2:0 [ComposedOp]: 240 params, indices=slice(480, 720, None), 2 sub-members
>>> Gxpi2:1 [ComposedOp]: 240 params, indices=slice(720, 960, None), 2 sub-members
>>> Gypi2:0 [ComposedOp]: 240 params, indices=slice(960, 1200, None), 2 sub-members
>>> Gypi2:1 [ComposedOp]: 240 params, indices=slice(1200, 1440, None), 2 sub-members
>>> Gii:0:1 [ComposedOp]: 240 params, indices=slice(1440, 1680, None), 2 sub-members
>>> Gecr:0:1 [ComposedOp]: 240 params, indices=slice(1680, 1920, None), 2 sub-members
>>> Gecr:1:0 [ComposedOp]: 240 params, indices=slice(1920, 2160, None), 2 sub-members


In [6]:
twoq_gstdesign = make_2q_edesign(twoq_target, maxmaxlen)

Initial Length Available Fiducial List: 1555
Length Available Fiducial List Dropped Identities and Duplicates: 281
Using greedy algorithm.
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.


/Users/rjmurr/Documents/pg-xfgst/repo/pygsti/forwardsims/mapforwardsim.py:745: ForwardSimulatorSuitabilityWarning: 
            Generating dense process matrix representations of circuits or gates
            can be inefficient and should be avoided for the purposes of forward
            simulation/calculation of circuit outcome probability distributions
            when using the MapForwardSimulator. This operator is small enough
            that it could be computed with MatrixForwardSimulator instead.
            
  _warnings.warn(msg, ForwardSimulatorSuitabilityWarning)


Acceptable candidate solution found.
Score: major=-16 minor=48.49999999999996, N: 16
Exiting greedy search.
Preparation fiducials:
['{}@(0,1)', 'Gecr:0:1@(0,1)', 'Gxpi2:1Gxpi2:1@(0,1)', 'Gxpi2:1Gecr:0:1Gxpi2:1@(0,1)', 'Gypi2:0Gxpi2:0Gypi2:1@(0,1)', 'Gxpi2:0Gecr:1:0Gypi2:0@(0,1)', 'Gypi2:1Gxpi2:0Gypi2:0@(0,1)', 'Gypi2:0Gecr:1:0Gxpi2:0@(0,1)', 'Gxpi2:0Gxpi2:1Gxpi2:1Gypi2:1@(0,1)', 'Gxpi2:1Gxpi2:1Gypi2:0Gypi2:1@(0,1)', 'Gxpi2:0Gxpi2:1Gecr:1:0Gypi2:1@(0,1)', 'Gecr:0:1Gypi2:0Gecr:1:0Gypi2:1@(0,1)', 'Gxpi2:0Gxpi2:0@(0,1)', 'Gxpi2:1Gypi2:1@(0,1)', 'Gypi2:1Gecr:0:1Gypi2:0@(0,1)', 'Gxpi2:1Gypi2:0Gecr:0:1Gypi2:1@(0,1)']
Score: 48.49999999999996
Complete initial fiducial set succeeds.
Now searching for best fiducial set.
Starting fiducial list optimization. Lower score is better.
Acceptable candidate solution found.
Score: major=-16 minor=16.30882352941179, N: 16
Exiting greedy search.
Measurement fiducials:
['{}@(0,1)', 'Gxpi2:0Gecr:1:0@(0,1)', 'Gecr:0:1Gypi2:0@(0,1)', 'Gxpi2:1Gypi2:1Gypi2:0@(0,

In [7]:
nq_target = make_nq_XYIECR_target_model(num_qubits=nq)

In [8]:
nq_pspec = nq_target.create_processor_spec()
temp = make_xfgst_design(
    nq_pspec, oneq_gstdesign, twoq_gstdesign, seed=0
)


In [9]:
len(temp.all_circuits_needing_data)

9622